# 00 — Data Recon

**Goal:** Find out what the real dataset looks like before anything is built, so the schema mapping in the handoff (sections 3.1 and 3.2) can be corrected.

**Rules:**
- Only read `data/model_input/`. The `data/ground_truth/` folder is answer-keys — the engine never reads it.
- No graph building yet.
- Every number printed is labelled **(verified)** once confirmed or **(plausible)** if not yet checked.

---

## Top 3 doubts before writing code

1. **Node/edge mapping may be wrong.** The handoff says status (b) — plausible but NOT verified. We print real distinct values to fix those tables.
2. **`security_controls` might be a list or a string** depending on asset type — need to check.
3. **Alias events** — 15% of events use a display name / hostname / IP instead of an asset id. We need real examples so the resolver knows what forms to handle.

In [1]:
from pathlib import Path
import json, sys, hashlib, statistics
from collections import Counter, defaultdict
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data_model/dataset/attack_path_dataset/data/model_input').is_dir())
sys.path.insert(0,str(ROOT))
DATA = ROOT/'data_model/dataset/attack_path_dataset/data'
MODEL = DATA/'model_input'
AUDIT = ROOT/'phase_a_audit'
ART = AUDIT/'artifacts'
ART.mkdir(parents=True,exist_ok=True)
TWIN_SOURCE_SHA = 'acfc662a54d5fcf02bd7adc896897946e6b35eae19cc500314000633f6c113f5'
checks = []
def check(name, condition):
    checks.append({'name':name,'passed':bool(condition)})
    print(('PASS: ' if condition else 'FAIL: ')+name)
    assert condition, name
def current_inputs():
    return {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(MODEL.glob('*')) if p.is_file()}
def save_stage(stage, payload):
    envelope = dict(run_id=run_id, input_hashes=current_inputs(), source_sha=TWIN_SOURCE_SHA, payload=payload)
    (ART/(stage+'.json')).write_text(json.dumps(envelope),encoding='utf-8')
def load_stage(stage):
    envelope = json.loads((ART/(stage+'.json')).read_text(encoding='utf-8'))
    check(stage+' current input fingerprints',envelope['input_hashes']==current_inputs())
    check(stage+' current source fingerprint',envelope['source_sha']==TWIN_SOURCE_SHA)
    check(stage+' current execution run',envelope['run_id']==run_id)
    return envelope['payload']
def report(stage, values):
    values['checks'] = checks.copy()
    (AUDIT/(stage+'_metrics.json')).write_text(json.dumps(values,indent=2),encoding='utf-8')
    print(json.dumps(values,indent=2))
def graph_pack(graphs):
    return {sid:dict(metadata=g.graph,nodes=[[n,d] for n,d in g.nodes(data=True)],edges=[[u,v,k,d] for u,v,k,d in g.edges(keys=True,data=True)]) for sid,g in graphs.items()}
def graph_unpack(packed):
    import networkx as nx
    graphs={}
    for sid,entry in packed.items():
        g=nx.MultiDiGraph()
        g.graph.update(entry['metadata'])
        g.add_nodes_from((n,d) for n,d in entry['nodes'])
        for u,v,k,d in entry['edges']:
            g.add_edge(u,v,key=k,**d)
        graphs[sid]=g
    return graphs

import uuid
run_id = str(uuid.uuid4())


In [2]:
import json
import os
from collections import Counter, defaultdict

# Adjust if running from a different CWD
DATASET_ROOT = str(DATA.parent)
MI = os.path.join(DATASET_ROOT, "data", "model_input")

def load_jsonl(path):
    """Load a .jsonl file — one JSON object per line — into a list of dicts."""
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

print("Loading all model_input files...")
scenarios     = load_jsonl(os.path.join(MI, "scenarios.jsonl"))
assets        = load_jsonl(os.path.join(MI, "assets.jsonl"))
relationships = load_jsonl(os.path.join(MI, "relationships.jsonl"))
events        = load_jsonl(os.path.join(MI, "events.jsonl"))
evidence      = load_jsonl(os.path.join(MI, "evidence.jsonl"))
interventions = load_jsonl(os.path.join(MI, "interventions.jsonl"))
what_if       = load_jsonl(os.path.join(MI, "what_if.jsonl"))
splits        = load_json(os.path.join(DATASET_ROOT, "data", "splits.json"))

print("Done.")

Loading all model_input files...
Done.


## 1 — Row counts vs handoff claims

In [3]:
# Handoff claims (dataset HANDOFF.md + handoff_attacktwin_v2_digital_twin.md §4)
HANDOFF_CLAIMS = {
    "scenarios":     60,
    "assets":        1576,
    "relationships": 3589,
    "events":        11253,
    "evidence":      11253,
    "interventions": 258,
    "what_if":       20,
}

actuals = {
    "scenarios":     len(scenarios),
    "assets":        len(assets),
    "relationships": len(relationships),
    "events":        len(events),
    "evidence":      len(evidence),
    "interventions": len(interventions),
    "what_if":       len(what_if),
}

print(f"{'File':<18} {'Handoff':>10} {'Actual':>10} {'Match?':>12}")
print("-" * 54)
all_match = True
for name, claimed in HANDOFF_CLAIMS.items():
    actual = actuals[name]
    ok = actual == claimed
    mark = "OK" if ok else f"DIFF {actual - claimed:+d}"
    if not ok:
        all_match = False
    print(f"{name:<18} {claimed:>10} {actual:>10} {mark:>12}")

print()
if all_match:
    print("All counts match the handoff. (verified)")
else:
    print("Some counts differ — rows marked DIFF. (verified)")

File                  Handoff     Actual       Match?
------------------------------------------------------
scenarios                  60         60           OK
assets                   1576       1576           OK
relationships            3589       3589           OK
events                  11253      11253           OK
evidence                11253      11253           OK
interventions             258        258           OK
what_if                    20         20           OK

All counts match the handoff. (verified)


## 2 — Distinct values and counts for key categorical fields

In [4]:
def count_field(rows, field):
    return Counter(r.get(field) for r in rows).most_common()

def print_counts(label, counts):
    print(f"\n--- {label} ---")
    for val, n in counts:
        print(f"  {str(val):<48} {n:>6}")

# Assets
print_counts("asset_type", count_field(assets, "asset_type"))
print_counts("zone",       count_field(assets, "zone"))
print_counts("exposure",   count_field(assets, "exposure"))


--- asset_type ---
  endpoint                                            312
  server                                              286
  user_identity                                       231
  application                                         195
  database                                            154
  privileged_identity                                 130
  cloud_identity                                      110
  security_infrastructure                              86
  external_entry                                       72

--- zone ---
  corporate                                           604
  server                                              466
  restricted                                          223
  dmz                                                 211
  external                                             72

--- exposure ---
  internal                                           1238
  public                                              184
  restricted        

In [5]:
# Relationships
print_counts("relationship_type", count_field(relationships, "relationship_type"))
print_counts("evidence_basis",    count_field(relationships, "evidence_basis"))


--- relationship_type ---
  NETWORK_REACHABILITY                               1239
  CAN_ASSUME_IDENTITY                                1173
  AUTHENTICATES_TO                                    596
  CAN_ACCESS_DATA                                     278
  TRUSTS_SERVICE                                      115
  HAS_PRIVILEGE                                        88
  DEPLOYS_TO                                           49
  ADMINISTERS                                          31
  CAN_MODIFY_CONFIGURATION                             20

--- evidence_basis ---
  configured                                         1545
  inferred                                           1059
  weak                                                745
  unsupported                                         240


In [6]:
# Events
print_counts("event source",  count_field(events, "source"))
print_counts("event_type",    count_field(events, "event_type"))


--- event source ---
  synthetic_cloud_audit                              2824
  synthetic_netflow                                  2057
  synthetic_idp_logs                                 1876
  synthetic_api_gateway                              1858
  synthetic_edr                                      1594
  synthetic_vuln_scan                                 701
  synthetic_secret_scan                               343

--- event_type ---
  connection_observed                                1527
  token_issued                                       1484
  suspicious_process                                  902
  malware_detected                                    692
  health_check                                        616
  access_blocked                                      538
  login_success                                       508
  api_access                                          497
  admin_login                                         454
  vulnerability_detected      

In [7]:
# security_controls — check if it's a list or a string
print("--- security_controls field type check ---")
type_counts = Counter(type(a.get("security_controls")).__name__ for a in assets)
for t, n in type_counts.items():
    print(f"  python type: {t:<12} count: {n}")

# Flatten and count all individual control values
all_controls = []
for a in assets:
    ctrl = a.get("security_controls", [])
    if isinstance(ctrl, list):
        all_controls.extend(ctrl)
    else:
        all_controls.append(ctrl)

print_counts("security_controls (distinct values)", Counter(all_controls).most_common())

--- security_controls field type check ---
  python type: list         count: 1576

--- security_controls (distinct values) ---
  mfa                                                 459
  audit_logging                                       458
  edr                                                 397
  patch_management                                    389
  network_acl                                         336
  rbac                                                293
  conditional_access                                  261
  rate_limiting                                       192
  disk_encryption                                     191
  waf                                                 176
  encryption_at_rest                                  108
  key_rotation                                         87
  just_in_time_access                                  79


## 3 — Splits: keys and first 3 entries per split

In [8]:
print("Keys in splits.json:", list(splits.keys()))
print()
for split_name in ("train", "validation", "test"):
    entries = splits.get(split_name, [])
    print(f"  {split_name}: {len(entries)} scenarios, first 3: {entries[:3]}")

Keys in splits.json: ['note', 'test', 'train', 'validation']

  train: 36 scenarios, first 3: ['scn_0001', 'scn_0004', 'scn_0005']
  validation: 12 scenarios, first 3: ['scn_0002', 'scn_0003', 'scn_0008']
  test: 12 scenarios, first 3: ['scn_0007', 'scn_0010', 'scn_0013']


## 4 — Raw example rows from each file (3 per file)

In [9]:
def show_examples(label, rows, n=3):
    print(f"\n=== {label} (first {n}) ===")
    for r in rows[:n]:
        print(json.dumps(r, indent=2))
        print()

show_examples("scenarios",     scenarios)
show_examples("assets",        assets)
show_examples("relationships", relationships)


=== scenarios (first 3) ===
{
  "asset_count": 35,
  "critical_targets": [
    "db_003"
  ],
  "entry_points": [
    "external_entry_001",
    "external_entry_002"
  ],
  "event_count": 168,
  "graph_version": "v1_baseline",
  "relationship_count": 52,
  "scenario_id": "scn_0001",
  "schema_version": "1.0",
  "synthetic": true,
  "time_window_end": "2026-10-03T00:00:00Z",
  "time_window_start": "2026-09-28T00:00:00Z",
  "trust_zones": [
    "corporate",
    "dmz",
    "external",
    "restricted",
    "server"
  ]
}

{
  "asset_count": 31,
  "critical_targets": [
    "db_002"
  ],
  "entry_points": [
    "external_entry_001"
  ],
  "event_count": 242,
  "graph_version": "v1_baseline",
  "relationship_count": 78,
  "scenario_id": "scn_0002",
  "schema_version": "1.0",
  "synthetic": true,
  "time_window_end": "2026-10-02T00:00:00Z",
  "time_window_start": "2026-09-27T00:00:00Z",
  "trust_zones": [
    "corporate",
    "dmz",
    "external",
    "restricted",
    "server"
  ]
}

{
  "as

In [10]:
show_examples("events",        events)
show_examples("evidence",      evidence)
show_examples("interventions", interventions)
show_examples("what_if",       what_if)


=== events (first 3) ===
{
  "event_id": "evt_000001",
  "event_type": "vulnerability_detected",
  "observed_fields": {
    "cvss_like": 5.4,
    "exploit_available": false,
    "finding_id": "SYN-VULN-2026-0392"
  },
  "payload_version": "1.0",
  "scenario_id": "scn_0001",
  "severity_raw": 5.4,
  "source": "synthetic_vuln_scan",
  "source_entity_raw": "external_entry_002",
  "target_entity_raw": "app-004.corp.example",
  "timestamp": "2026-09-28T00:15:00Z"
}

{
  "event_id": "evt_000002",
  "event_type": "connection_observed",
  "observed_fields": {
    "bytes_out": 29121,
    "dst_port": 443,
    "first_seen_pair": false,
    "protocol": "tcp"
  },
  "payload_version": "1.0",
  "scenario_id": "scn_0001",
  "severity_raw": null,
  "source": "synthetic_netflow",
  "source_entity_raw": "APP-004.CORP.EXAMPLE",
  "target_entity_raw": "host_004",
  "timestamp": 1790554500
}

{
  "event_id": "evt_000003",
  "event_type": "token_issued",
  "observed_fields": {
    "scope_count": 7,
    "to

## 5 — 5 events that use an alias instead of an asset id

In [11]:
# A full set of all asset ids across all scenarios for quick membership checks
all_asset_ids = {(a["scenario_id"], a["asset_id"]) for a in assets}

# Alias = source_entity_raw is not null AND is not a known asset id
alias_src_events = [
    e for e in events
    if e.get("source_entity_raw") is not None
    and (e["scenario_id"], e["source_entity_raw"]) not in all_asset_ids
]

alias_tgt_events = [
    e for e in events
    if e.get("target_entity_raw") is not None
    and (e["scenario_id"], e["target_entity_raw"]) not in all_asset_ids
]

print(f"Events with alias source_entity_raw: {len(alias_src_events)} "
      f"({100*len(alias_src_events)/len(events):.1f}%)")
print(f"Events with alias target_entity_raw: {len(alias_tgt_events)} "
      f"({100*len(alias_tgt_events)/len(events):.1f}%)")

print("\n5 alias source examples (shows what forms to handle):")
for e in alias_src_events[:5]:
    print({
        "event_id":          e["event_id"],
        "scenario_id":       e["scenario_id"],
        "event_type":        e["event_type"],
        "source_entity_raw": e["source_entity_raw"],
        "target_entity_raw": e.get("target_entity_raw"),
        "source":            e["source"],
    })

Events with alias source_entity_raw: 1738 (15.4%)
Events with alias target_entity_raw: 1361 (12.1%)

5 alias source examples (shows what forms to handle):
{'event_id': 'evt_000002', 'scenario_id': 'scn_0001', 'event_type': 'connection_observed', 'source_entity_raw': 'APP-004.CORP.EXAMPLE', 'target_entity_raw': 'host_004', 'source': 'synthetic_netflow'}
{'event_id': 'evt_000010', 'scenario_id': 'scn_0001', 'event_type': 'suspicious_process', 'source_entity_raw': '10.0.3.4', 'target_entity_raw': 'SERVER-002.CORP.EXAMPLE', 'source': 'synthetic_edr'}
{'event_id': 'evt_000011', 'scenario_id': 'scn_0001', 'event_type': 'service_error', 'source_entity_raw': 'server-001.corp.example', 'target_entity_raw': None, 'source': 'synthetic_api_gateway'}
{'event_id': 'evt_000013', 'scenario_id': 'scn_0001', 'event_type': 'api_access', 'source_entity_raw': 'kavor.drezenvor@corp.example', 'target_entity_raw': 'db_003', 'source': 'synthetic_api_gateway'}
{'event_id': 'evt_000018', 'scenario_id': 'scn_0001

## 6 — Referential integrity: do edge endpoints exist in their scenario's assets?

In [12]:
# Build a set of (scenario_id, asset_id) pairs
asset_pairs = set((a["scenario_id"], a["asset_id"]) for a in assets)

missing_src = [r for r in relationships if (r["scenario_id"], r["source_id"]) not in asset_pairs]
missing_tgt = [r for r in relationships if (r["scenario_id"], r["target_id"]) not in asset_pairs]

print(f"Edges with missing source_id in scenario: {len(missing_src)}")
print(f"Edges with missing target_id in scenario: {len(missing_tgt)}")

if not missing_src and not missing_tgt:
    print("\nPASS — All edge endpoints exist in their scenario. (verified)")
else:
    print("\nFAIL — showing first 5 broken edges:")
    for r in (missing_src + missing_tgt)[:5]:
        print(json.dumps(r, indent=2))

Edges with missing source_id in scenario: 0
Edges with missing target_id in scenario: 0

PASS — All edge endpoints exist in their scenario. (verified)


## 7 — Prior probability ranges by evidence_basis

In [13]:
# Handoff says: configured 0.70-0.95, inferred 0.25-0.60, weak 0.10-0.30, unsupported 0.02-0.10
EXPECTED_RANGES = {
    "configured":  (0.70, 0.95),
    "inferred":    (0.25, 0.60),
    "weak":        (0.10, 0.30),
    "unsupported": (0.02, 0.10),
}

by_basis = defaultdict(list)
for r in relationships:
    by_basis[r["evidence_basis"]].append(r["prior_probability"])

print(f"{'basis':<14} {'count':>6} {'min':>7} {'max':>7} {'mean':>7} {'expected_range':>17} {'ok?':>6}")
print("-" * 70)
for basis, vals in sorted(by_basis.items()):
    lo, hi = min(vals), max(vals)
    mean = sum(vals) / len(vals)
    exp = EXPECTED_RANGES.get(basis, (None, None))
    ok = "OK" if exp[0] is not None and lo >= exp[0] - 0.01 and hi <= exp[1] + 0.01 else "CHECK"
    print(f"{basis:<14} {len(vals):>6} {lo:>7.3f} {hi:>7.3f} {mean:>7.3f} {str(exp):>17} {ok:>6}")

basis           count     min     max    mean    expected_range    ok?
----------------------------------------------------------------------
configured       1545   0.700   0.950   0.827       (0.7, 0.95)     OK
inferred         1059   0.250   0.600   0.426       (0.25, 0.6)     OK
unsupported       240   0.020   0.100   0.061       (0.02, 0.1)     OK
weak              745   0.101   0.300   0.197        (0.1, 0.3)     OK


## 8 — Timestamp and severity format survey

In [14]:
# Confirms the multi-format warning in the handoff — parser must branch per source
ts_types  = defaultdict(set)
sev_types = defaultdict(set)
sev_samples = defaultdict(set)

for e in events:
    src = e["source"]
    ts_types[src].add(type(e["timestamp"]).__name__)
    sev_types[src].add(type(e["severity_raw"]).__name__)
    if len(sev_samples[src]) < 6:
        sev_samples[src].add(repr(e["severity_raw"]))

print(f"{'source':<28} {'ts_type':<10} {'sev_type':<14} sev_sample_values")
print("-" * 90)
for src in sorted(ts_types):
    ts  = list(ts_types[src])[0]
    sev = list(sev_types[src])[0]
    print(f"{src:<28} {ts:<10} {sev:<14} {sorted(sev_samples[src])}")

source                       ts_type    sev_type       sev_sample_values
------------------------------------------------------------------------------------------
synthetic_api_gateway        int        str            ["'error'", "'notice'", "'warning'"]
synthetic_cloud_audit        str        str            ["'HIGH'", "'LOW'", "'MEDIUM'"]
synthetic_edr                str        int            ['10', '3', '6', '7', '8', '9']
synthetic_idp_logs           str        str            ["'alert'", "'info'", "'warn'"]
synthetic_netflow            int        NoneType       ['None']
synthetic_secret_scan        str        str            ["'high'", "'low'", "'medium'"]
synthetic_vuln_scan          str        float          ['1.0', '2.5', '3.0', '4.4', '5.1', '5.4']


Historical hardcoded conclusions removed from this executed copy. The audit summary below is derived from current input rows; the original notebook is preserved in phase_a_audit/before_audit/.

## Executed Phase A audit: complete data values and quality checks
Alias-like means non-null and not an exact asset ID in the same scenario. Known aliases and ambiguous aliases are distinguished from unmatched references; unmatched does not prove a real-world entity is invalid.

In [15]:
tables = dict(scenarios=scenarios,assets=assets,relationships=relationships,events=events,evidence=evidence,interventions=interventions,what_if=what_if)
fields = {'asset_type':(assets,'asset_type'),'relationship_type':(relationships,'relationship_type'),'zone':(assets,'zone'),'exposure':(assets,'exposure'),'event_source':(events,'source'),'event_type':(events,'event_type'),'evidence_basis':(relationships,'evidence_basis')}
distributions = {name:dict(Counter(r[key] for r in rows)) for name,(rows,key) in fields.items()}
asset_index = {(a['scenario_id'],a['asset_id']):a for a in assets}
alias_index=defaultdict(lambda:defaultdict(set))
for a in assets:
    for field in ('asset_id','display_name','email','hostname','ip_address'):
        if a.get(field): alias_index[a['scenario_id']][a[field].lower()].add(a['asset_id'])
references={}
for role in ('source','target'):
    counts=Counter()
    examples=[]
    for e in events:
        ref=e.get(role+'_entity_raw')
        if ref is None: kind='null'
        elif (e['scenario_id'],ref) in asset_index: kind='exact_id'
        else:
            ids=alias_index[e['scenario_id']].get(ref.lower(),set())
            kind='alias_unique' if len(ids)==1 else 'alias_ambiguous' if ids else 'unmatched_non_null'
            if len(examples)<5: examples.append(dict(event_id=e['event_id'],raw=ref,candidates=sorted(ids),classification=kind))
        counts[kind]+=1
    alias_like=sum(counts[k] for k in ('alias_unique','alias_ambiguous','unmatched_non_null'))
    references[role]=dict(counts=counts,alias_like=alias_like,percent_all_events=100*alias_like/len(events),percent_non_null=100*alias_like/(len(events)-counts['null']),examples=examples)
id_fields={'scenarios':'scenario_id','assets':'asset_id','relationships':'edge_id','events':'event_id','evidence':'event_id','interventions':'intervention_id','what_if':'what_if_id'}
duplicates={}
for name,rows in tables.items():
    field=id_fields[name]
    scoped=Counter((r['scenario_id'],r[field]) for r in rows)
    global_counts=Counter(r[field] for r in rows)
    id_scenarios=defaultdict(set)
    for r in rows:id_scenarios[r[field]].add(r['scenario_id'])
    duplicates[name]=dict(within_scenario_duplicate_rows=sum(n-1 for n in scoped.values()),global_duplicate_rows=sum(n-1 for n in global_counts.values()),distinct_ids_reused_across_scenarios=sum(len(s)>1 for s in id_scenarios.values()))
formats={}
for src in sorted({e['source'] for e in events}):
    rows=[e for e in events if e['source']==src]
    formats[src]=dict(count=len(rows),timestamp_python_types=dict(Counter(type(e['timestamp']).__name__ for e in rows)),timestamp_examples=[r['timestamp'] for r in rows[:3]],severity_python_types=dict(Counter(type(e['severity_raw']).__name__ for e in rows)),severity_values=sorted({str(r['severity_raw']) for r in rows}),timestamp_format='epoch milliseconds' if src=='synthetic_api_gateway' else 'epoch seconds' if src=='synthetic_netflow' else '%Y-%m-%d %H:%M:%S UTC' if src=='synthetic_edr' else '%Y-%m-%dT%H:%M:%SZ')
broken=[r for r in relationships if (r['scenario_id'],r['source_id']) not in asset_index or (r['scenario_id'],r['target_id']) not in asset_index]
split_sets={k:set(splits[k]) for k in ('train','validation','test')}
check('No broken relationship endpoints',not broken)
check('No scenario-scoped duplicate IDs',all(d['within_scenario_duplicate_rows']==0 for d in duplicates.values()))
check('Splits are pairwise disjoint',not any(split_sets[a]&split_sets[b] for a,b in [('train','validation'),('train','test'),('validation','test')]))
check('Splits cover exactly input scenarios',set.union(*split_sets.values())=={s['scenario_id'] for s in scenarios})
metrics=dict(row_counts={k:len(v) for k,v in tables.items()},distinct_counts={k:len(v) for k,v in distributions.items()},distributions=distributions,security_controls_types=dict(Counter(type(a.get('security_controls')).__name__ for a in assets)),control_item_types=dict(Counter(type(c).__name__ for a in assets for c in a['security_controls'])),references=references,broken_relationships=broken,id_duplicates=duplicates,splits={k:splits[k] for k in split_sets},formats=formats,handoff_count_mismatches={k:dict(claimed=HANDOFF_CLAIMS[k],actual=len(v)) for k,v in tables.items() if HANDOFF_CLAIMS[k]!=len(v)},prior_ranges={k:dict(min=min(v),max=max(v),mean=sum(v)/len(v)) for k,v in by_basis.items()})
save_stage('00_recon',dict(tables=tables,splits=splits))
report('00',metrics)


PASS: No broken relationship endpoints


PASS: No scenario-scoped duplicate IDs
PASS: Splits are pairwise disjoint
PASS: Splits cover exactly input scenarios


{
  "row_counts": {
    "scenarios": 60,
    "assets": 1576,
    "relationships": 3589,
    "events": 11253,
    "evidence": 11253,
    "interventions": 258,
    "what_if": 20
  },
  "distinct_counts": {
    "asset_type": 9,
    "relationship_type": 9,
    "zone": 5,
    "exposure": 4,
    "event_source": 7,
    "event_type": 26,
    "evidence_basis": 4
  },
  "distributions": {
    "asset_type": {
      "user_identity": 231,
      "privileged_identity": 130,
      "endpoint": 312,
      "server": 286,
      "application": 195,
      "database": 154,
      "cloud_identity": 110,
      "security_infrastructure": 86,
      "external_entry": 72
    },
    "relationship_type": {
      "AUTHENTICATES_TO": 596,
      "CAN_ASSUME_IDENTITY": 1173,
      "NETWORK_REACHABILITY": 1239,
      "TRUSTS_SERVICE": 115,
      "HAS_PRIVILEGE": 88,
      "ADMINISTERS": 31,
      "CAN_MODIFY_CONFIGURATION": 20,
      "CAN_ACCESS_DATA": 278,
      "DEPLOYS_TO": 49
    },
    "zone": {
      "corporate": 60